In [13]:
# ============================================================================
# ANIME CLUSTER RATING DISTRIBUTION ANALYSIS - JUPYTER NOTEBOOK VERSION
# ============================================================================

# FILE PATHS - MODIFY THESE TO POINT TO YOUR DATA FILES
# ============================================================================
ANIME_DATA_PATH = r"..\data\anime.csv"                 # Path to your anime metadata CSV  
CLUSTER_ASSIGNMENTS_PATH = r"..\data\final_clusters\spectral_13_clusters.csv"    # Path to your cluster assignments CSV with columns: anime_id,cluster
# NOTE: Your cluster assignments CSV should use MAL_ID to match the anime.csv file
# Expected format: MAL_ID,cluster


# ============================================================================
# IMPORTS AND SETUP
# ============================================================================

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import kruskal, mannwhitneyu, levene, shapiro, chi2_contingency
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All packages imported successfully!")
print("📁 File paths configured:")
print(f"   Anime Data (with score distributions): {ANIME_DATA_PATH}")
print(f"   Cluster Assignments: {CLUSTER_ASSIGNMENTS_PATH}")


✅ All packages imported successfully!
📁 File paths configured:
   Anime Data (with score distributions): ..\data\anime.csv
   Cluster Assignments: ..\data\final_clusters\spectral_13_clusters.csv


# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

In [17]:
def load_anime_data():
    """Load anime data with score distributions and cluster assignments."""
    print("📊 Loading anime data with score distributions...")
    
    # Load anime data
    try:
        anime_df = pd.read_csv(ANIME_DATA_PATH)
        print(f"✅ Loaded {len(anime_df):,} anime records")
        
        # Check for required columns
        score_columns = [f'score_{str(i).zfill(2)}_count' for i in range(1, 11)]
        print(f"📋 Anime file columns found: {list(anime_df.columns)[:10]}... (showing first 10)")
        
        missing_cols = [col for col in score_columns if col not in anime_df.columns]
        if missing_cols:
            print(f"⚠️  Warning: Missing score columns: {missing_cols}")
        else:
            print(f"✅ All required score columns found: Score-1 through Score-10")
        
        # Clean data - remove anime with no score distributions
        anime_df = anime_df.dropna(subset=['anime_id'])
        
        # Filter to anime that have at least some ratings
        anime_df['total_ratings'] = anime_df[score_columns].fillna(0).sum(axis=1)
        anime_df = anime_df[anime_df['total_ratings'] > 0]
        print(f"📊 After filtering: {len(anime_df):,} anime with valid rating distributions")
        
    except FileNotFoundError:
        print(f"❌ Error: Could not find anime data file at {ANIME_DATA_PATH}")
        return None, None
    
    # Load cluster assignments
    try:
        cluster_df = pd.read_csv(CLUSTER_ASSIGNMENTS_PATH)
        print(f"✅ Loaded cluster assignments for {len(cluster_df):,} anime")
        print(f"📋 Cluster file columns: {list(cluster_df.columns)}")
        
        # Check for anime_id column
        if 'anime_id' not in cluster_df.columns:
            print("❌ Error: Cluster file must have 'anime_id' column")
            print(f"   Found columns: {list(cluster_df.columns)}")
            return None, None
            
        # Merge anime data with cluster assignments
        merged_df = anime_df.merge(cluster_df, on='anime_id', how='inner')
        
        if len(merged_df) == 0:
            print("❌ Error: No matching anime between anime data and cluster assignments!")
            print("Check that anime_id values match between datasets.")
            return None, None
        
        print(f"✅ Successfully merged data for {len(merged_df):,} anime across {merged_df['cluster'].nunique()} clusters")
        
        return merged_df, cluster_df
        
    except FileNotFoundError:
        print(f"❌ Error: Could not find cluster assignments file at {CLUSTER_ASSIGNMENTS_PATH}")
        return None, None

# Load the data
anime_data, cluster_assignments = load_anime_data()

📊 Loading anime data with score distributions...
✅ Loaded 13,379 anime records
📋 Anime file columns found: ['anime_id', 'anime_url', 'title', 'synopsis', 'main_pic', 'type', 'source_type', 'num_episodes', 'status', 'start_date']... (showing first 10)
✅ All required score columns found: Score-1 through Score-10
📊 After filtering: 10,714 anime with valid rating distributions
✅ Loaded cluster assignments for 2,000 anime
📋 Cluster file columns: ['anime_id', 'cluster']
✅ Successfully merged data for 2,000 anime across 13 clusters


# ============================================================================
# ANIME CLUSTER RATING VARIANCE ANALYSIS CLASS
# ============================================================================

In [22]:
class AnimeClusterVarianceAnalysis:
    def __init__(self, anime_data):
        """
        Initialize the analysis with anime data containing score distributions.
        
        Parameters:
        anime_data (pd.DataFrame): DataFrame with anime_id, cluster, and score_XX_count columns
        """
        self.anime_data = anime_data
        self.score_columns = ['score_10_count', 'score_09_count', 'score_08_count', 'score_07_count', 'score_06_count', 
                             'score_05_count', 'score_04_count', 'score_03_count', 'score_02_count', 'score_01_count']
        self.rating_values = list(range(1, 11))
        
        print(f"🎯 Variance Analysis initialized:")
        print(f"   Total anime analyzed: {len(self.anime_data):,}")
        print(f"   Total individual ratings: {self.anime_data[self.score_columns].fillna(0).sum().sum():,.0f}")
        print(f"   Clusters: {self.anime_data['cluster'].nunique()}")
        print(f"   Average ratings per anime: {self.anime_data[self.score_columns].fillna(0).sum(axis=1).mean():.0f}")
        print("   Memory-efficient mode: Working with aggregated distributions")
    
    def _calculate_stats_from_counts(self, score_counts_row):
        """Calculate statistics directly from score count data without expanding to individual records."""
        # Score mapping
        score_mapping = {
            'score_01_count': 1, 'score_02_count': 2, 'score_03_count': 3, 'score_04_count': 4, 'score_05_count': 5,
            'score_06_count': 6, 'score_07_count': 7, 'score_08_count': 8, 'score_09_count': 9, 'score_10_count': 10
        }
        
        # Get counts and ratings
        counts = []
        ratings = []
        total_count = 0
        
        for score_col, rating in score_mapping.items():
            count = score_counts_row.get(score_col, 0)
            if pd.notna(count) and count > 0:
                counts.append(int(count))
                ratings.append(rating)
                total_count += int(count)
        
        if total_count == 0:
            return None
        
        # Calculate weighted statistics
        # Mean = sum(rating * count) / total_count
        mean = sum(r * c for r, c in zip(ratings, counts)) / total_count
        
        # Variance = sum(count * (rating - mean)^2) / (total_count - 1)
        if total_count > 1:
            variance = sum(c * (r - mean)**2 for r, c in zip(ratings, counts)) / (total_count - 1)
            std = np.sqrt(variance)
        else:
            variance = 0
            std = 0
        
        # Median (approximate using interpolation)
        cumulative = 0
        median = None
        for r, c in zip(ratings, counts):
            cumulative += c
            if cumulative >= total_count / 2:
                median = r
                break
        
        # Quartiles (approximate)
        cumulative = 0
        q1 = None
        q3 = None
        for r, c in zip(ratings, counts):
            cumulative += c
            if q1 is None and cumulative >= total_count * 0.25:
                q1 = r
            if cumulative >= total_count * 0.75:
                q3 = r
                break
        
        # Skewness and kurtosis (using moment formulas)
        if total_count > 2 and std > 0:
            # Third moment (skewness)
            m3 = sum(c * (r - mean)**3 for r, c in zip(ratings, counts)) / total_count
            skewness = m3 / (std**3)
            
            # Fourth moment (kurtosis)
            m4 = sum(c * (r - mean)**4 for r, c in zip(ratings, counts)) / total_count
            kurtosis = (m4 / (std**4)) - 3
        else:
            skewness = np.nan
            kurtosis = np.nan
        
        return {
            'total_ratings': total_count,
            'mean': mean,
            'median': median,
            'std': std,
            'variance': variance,
            'min': min(ratings) if ratings else np.nan,
            'max': max(ratings) if ratings else np.nan,
            'q1': q1,
            'q3': q3,
            'iqr': (q3 - q1) if (q1 is not None and q3 is not None) else np.nan,
            'skewness': skewness,
            'kurtosis': kurtosis,
            'cv': (std / mean * 100) if mean != 0 else 0,
        }
    
    def _get_cluster_ratings_sample(self, cluster, max_sample=10000):
        """Get a sample of individual ratings for a cluster (for statistical tests)."""
        cluster_anime = self.anime_data[self.anime_data['cluster'] == cluster]
        
        score_mapping = {
            'score_01_count': 1, 'score_02_count': 2, 'score_03_count': 3, 'score_04_count': 4, 'score_05_count': 5,
            'score_06_count': 6, 'score_07_count': 7, 'score_08_count': 8, 'score_09_count': 9, 'score_10_count': 10
        }
        
        all_ratings = []
        total_ratings = 0
        
        # First pass: count total ratings
        for _, anime_row in cluster_anime.iterrows():
            for score_col, rating in score_mapping.items():
                count = anime_row.get(score_col, 0)
                if pd.notna(count) and count > 0:
                    total_ratings += int(count)
        
        # Calculate sampling ratio if needed
        if total_ratings > max_sample:
            sample_ratio = max_sample / total_ratings
            print(f"   Sampling {max_sample:,} from {total_ratings:,} ratings for cluster {cluster}")
        else:
            sample_ratio = 1.0
        
        # Second pass: collect ratings (with sampling if needed)
        for _, anime_row in cluster_anime.iterrows():
            for score_col, rating in score_mapping.items():
                count = anime_row.get(score_col, 0)
                if pd.notna(count) and count > 0:
                    # Sample the count if needed
                    sampled_count = int(count * sample_ratio) if sample_ratio < 1.0 else int(count)
                    all_ratings.extend([rating] * sampled_count)
        
        return np.array(all_ratings)
    
    def calculate_anime_level_statistics(self):
        """Calculate statistics at the anime level using score distributions."""
        anime_stats = []
        
        for _, anime_row in self.anime_data.iterrows():
            anime_id = anime_row['anime_id']
            cluster = anime_row['cluster']
            anime_name = anime_row.get('title', f'Anime_{anime_id}')
            
            # Calculate stats using the efficient method
            stats_result = self._calculate_stats_from_counts(anime_row)
            
            if stats_result is None:  # Skip anime with no ratings
                continue
            
            # Combine basic info with calculated stats
            stats_dict = {
                'anime_id': anime_id,
                'cluster': cluster,
                'anime_name': anime_name,
                **stats_result
            }
            
            # Add percentage distributions and polarization metrics
            score_mapping = {
                'score_01_count': 1, 'score_02_count': 2, 'score_03_count': 3, 'score_04_count': 4, 'score_05_count': 5,
                'score_06_count': 6, 'score_07_count': 7, 'score_08_count': 8, 'score_09_count': 9, 'score_10_count': 10
            }
            
            total_ratings = stats_result['total_ratings']
            
            # Rating distribution percentages
            for score_col, rating in score_mapping.items():
                count = anime_row.get(score_col, 0)
                count = count if pd.notna(count) else 0
                stats_dict[f'pct_rating_{rating}'] = (count / total_ratings * 100) if total_ratings > 0 else 0
            
            # Polarization metrics
            high_ratings = sum([anime_row.get(f'score_{r:02d}_count', 0) for r in [8, 9, 10]])
            low_ratings = sum([anime_row.get(f'score_{r:02d}_count', 0) for r in [1, 2, 3]])
            mid_ratings = sum([anime_row.get(f'score_{r:02d}_count', 0) for r in [4, 5, 6, 7]])
            
            stats_dict['pct_high_ratings'] = (high_ratings / total_ratings * 100) if total_ratings > 0 else 0
            stats_dict['pct_low_ratings'] = (low_ratings / total_ratings * 100) if total_ratings > 0 else 0
            stats_dict['pct_mid_ratings'] = (mid_ratings / total_ratings * 100) if total_ratings > 0 else 0
            stats_dict['polarization_index'] = (high_ratings + low_ratings) / total_ratings if total_ratings > 0 else 0
            
            anime_stats.append(stats_dict)
        
        return pd.DataFrame(anime_stats)
    
    def cluster_level_analysis(self):
        """Analyze rating patterns at the cluster level."""
        cluster_summary = []
        
        for cluster in sorted(self.anime_data['cluster'].unique()):
            cluster_anime = self.anime_data[self.anime_data['cluster'] == cluster]
            
            if len(cluster_anime) == 0:
                continue
            
            # Calculate cluster-wide statistics by aggregating all anime in cluster
            cluster_total_counts = {}
            score_mapping = {
                'score_01_count': 1, 'score_02_count': 2, 'score_03_count': 3, 'score_04_count': 4, 'score_05_count': 5,
                'score_06_count': 6, 'score_07_count': 7, 'score_08_count': 8, 'score_09_count': 9, 'score_10_count': 10
            }
            
            # Sum all rating counts across anime in this cluster
            for score_col in score_mapping.keys():
                cluster_total_counts[score_col] = cluster_anime[score_col].fillna(0).sum()
            
            # Calculate cluster-level stats using aggregated counts
            cluster_stats_result = self._calculate_stats_from_counts(cluster_total_counts)
            
            if cluster_stats_result is None:
                continue
            
            # Basic cluster statistics
            cluster_stats = {
                'cluster': cluster,
                'num_anime': len(cluster_anime),
                'total_ratings': cluster_stats_result['total_ratings'],
                'avg_ratings_per_anime': cluster_stats_result['total_ratings'] / len(cluster_anime),
                'mean_rating': cluster_stats_result['mean'],
                'median_rating': cluster_stats_result['median'],
                'std_rating': cluster_stats_result['std'],
                'variance_rating': cluster_stats_result['variance'],
                'cv_rating': cluster_stats_result['cv'],
                'skewness': cluster_stats_result['skewness'],
                'kurtosis': cluster_stats_result['kurtosis'],
            }
            
            # Rating distribution for cluster
            total_cluster_ratings = cluster_stats_result['total_ratings']
            
            for score_col, rating in score_mapping.items():
                count = cluster_total_counts[score_col]
                cluster_stats[f'pct_rating_{rating}'] = (count / total_cluster_ratings * 100) if total_cluster_ratings > 0 else 0
            
            # Variance within cluster (anime-to-anime variance)
            anime_means = []
            anime_variances = []
            
            for _, anime_row in cluster_anime.iterrows():
                anime_stats = self._calculate_stats_from_counts(anime_row)
                if anime_stats is not None:
                    anime_means.append(anime_stats['mean'])
                    anime_variances.append(anime_stats['variance'])
            
            if anime_means:
                cluster_stats['between_anime_variance'] = np.var(anime_means, ddof=1) if len(anime_means) > 1 else 0
                cluster_stats['avg_within_anime_variance'] = np.mean(anime_variances)
                cluster_stats['mean_of_anime_means'] = np.mean(anime_means)
                cluster_stats['std_of_anime_means'] = np.std(anime_means, ddof=1) if len(anime_means) > 1 else 0
            
            cluster_summary.append(cluster_stats)
        
        return pd.DataFrame(cluster_summary)
    
    def statistical_tests(self):
        """Perform comprehensive statistical tests comparing clusters."""
        clusters = sorted(self.anime_data['cluster'].unique())
        
        if len(clusters) < 2:
            print("⚠️  Warning: Need at least 2 clusters for statistical tests")
            return {}
        
        results = {}
        print("📊 Running statistical tests with sampled data...")
        
        # Get sampled rating data by cluster (memory-efficient)
        cluster_ratings = {}
        cluster_anime_means = {}
        
        for cluster in clusters:
            # Sample individual ratings for this cluster
            cluster_ratings[cluster] = self._get_cluster_ratings_sample(cluster, max_sample=5000)
            
            # Calculate mean rating for each anime in cluster
            cluster_anime = self.anime_data[self.anime_data['cluster'] == cluster]
            anime_means = []
            for _, anime_row in cluster_anime.iterrows():
                anime_stats = self._calculate_stats_from_counts(anime_row)
                if anime_stats is not None:
                    anime_means.append(anime_stats['mean'])
            cluster_anime_means[cluster] = np.array(anime_means)
        
        # 1. ANOVA on individual ratings
        try:
            ratings_by_cluster = [cluster_ratings[c] for c in clusters if len(cluster_ratings[c]) > 0]
            if len(ratings_by_cluster) >= 2:
                f_stat, p_value = stats.f_oneway(*ratings_by_cluster)
                results['anova_individual_ratings'] = {
                    'f_statistic': f_stat,
                    'p_value': p_value,
                    'significant': p_value < 0.05,
                    'interpretation': 'Cluster rating distributions are significantly different' if p_value < 0.05 
                                   else 'No significant difference between cluster rating distributions'
                }
        except Exception as e:
            print(f"⚠️  Could not perform ANOVA on individual ratings: {e}")
        
        # 2. ANOVA on anime means
        try:
            means_by_cluster = [cluster_anime_means[c] for c in clusters if len(cluster_anime_means[c]) > 0]
            if len(means_by_cluster) >= 2:
                f_stat, p_value = stats.f_oneway(*means_by_cluster)
                results['anova_anime_means'] = {
                    'f_statistic': f_stat,
                    'p_value': p_value,
                    'significant': p_value < 0.05,
                    'interpretation': 'Cluster anime means are significantly different' if p_value < 0.05 
                                   else 'No significant difference between cluster anime means'
                }
        except Exception as e:
            print(f"⚠️  Could not perform ANOVA on anime means: {e}")
        
        # 3. Kruskal-Wallis test (non-parametric)
        try:
            h_stat, p_value = kruskal(*ratings_by_cluster)
            results['kruskal_wallis'] = {
                'h_statistic': h_stat,
                'p_value': p_value,
                'significant': p_value < 0.05,
                'interpretation': 'Cluster distributions are significantly different' if p_value < 0.05 
                               else 'No significant difference between cluster distributions'
            }
        except Exception as e:
            print(f"⚠️  Could not perform Kruskal-Wallis test: {e}")
        
        # 4. Levene's test for equal variances
        try:
            levene_stat, p_value = levene(*ratings_by_cluster)
            results['levene_test'] = {
                'statistic': levene_stat,
                'p_value': p_value,
                'equal_variances': p_value >= 0.05,
                'interpretation': 'Equal variances across clusters' if p_value >= 0.05 
                               else 'Unequal variances across clusters (some clusters more variable)'
            }
        except Exception as e:
            print(f"⚠️  Could not perform Levene test: {e}")
        
        # 5. Chi-square test for rating distribution patterns
        try:
            # Create contingency table using actual counts
            contingency_data = []
            score_mapping = {
                'score_01_count': 1, 'score_02_count': 2, 'score_03_count': 3, 'score_04_count': 4, 'score_05_count': 5,
                'score_06_count': 6, 'score_07_count': 7, 'score_08_count': 8, 'score_09_count': 9, 'score_10_count': 10
            }
            
            for cluster in clusters:
                cluster_anime = self.anime_data[self.anime_data['cluster'] == cluster]
                cluster_dist = []
                
                for score_col, rating in score_mapping.items():
                    total_count = cluster_anime[score_col].fillna(0).sum()
                    cluster_dist.append(int(total_count))
                
                contingency_data.append(cluster_dist)
            
            contingency_table = np.array(contingency_data)
            chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)
            
            results['chi_square_test'] = {
                'chi2_statistic': chi2_stat,
                'p_value': p_value,
                'degrees_of_freedom': dof,
                'significant': p_value < 0.05,
                'interpretation': 'Rating distribution patterns differ significantly between clusters' if p_value < 0.05 
                               else 'No significant difference in rating distribution patterns'
            }
        except Exception as e:
            print(f"⚠️  Could not perform Chi-square test: {e}")
        
        print("✅ Statistical tests completed")
        return results
    
    def variance_decomposition(self):
        """Decompose total variance into between-cluster and within-cluster components."""
        print("📊 Calculating variance decomposition...")
        
        # Calculate overall statistics using aggregated approach
        all_cluster_totals = {}
        score_mapping = {
            'score_01_count': 1, 'score_02_count': 2, 'score_03_count': 3, 'score_04_count': 4, 'score_05_count': 5,
            'score_06_count': 6, 'score_07_count': 7, 'score_08_count': 8, 'score_09_count': 9, 'score_10_count': 10
        }
        
        # Aggregate all rating counts across all anime
        for score_col in score_mapping.keys():
            all_cluster_totals[score_col] = self.anime_data[score_col].fillna(0).sum()
        
        overall_stats = self._calculate_stats_from_counts(all_cluster_totals)
        overall_mean = overall_stats['mean']
        total_variance = overall_stats['variance']
        
        # Calculate between-cluster variance
        cluster_means = {}
        cluster_sizes = {}
        
        for cluster in self.anime_data['cluster'].unique():
            cluster_anime = self.anime_data[self.anime_data['cluster'] == cluster]
            
            # Aggregate cluster totals
            cluster_totals = {}
            for score_col in score_mapping.keys():
                cluster_totals[score_col] = cluster_anime[score_col].fillna(0).sum()
            
            cluster_stats = self._calculate_stats_from_counts(cluster_totals)
            if cluster_stats is not None:
                cluster_means[cluster] = cluster_stats['mean']
                cluster_sizes[cluster] = cluster_stats['total_ratings']
        
        between_cluster_var = 0
        total_n = sum(cluster_sizes.values())
        
        for cluster in cluster_means:
            cluster_mean = cluster_means[cluster]
            cluster_n = cluster_sizes[cluster]
            between_cluster_var += cluster_n * (cluster_mean - overall_mean) ** 2
        
        between_cluster_var = between_cluster_var / (total_n - 1) if total_n > 1 else 0
        within_cluster_var = total_variance - between_cluster_var
        
        # Calculate percentage of variance explained by clustering
        percent_between = (between_cluster_var / total_variance) * 100 if total_variance > 0 else 0
        percent_within = (within_cluster_var / total_variance) * 100 if total_variance > 0 else 0
        
        return {
            'total_variance': total_variance,
            'between_cluster_variance': between_cluster_var,
            'within_cluster_variance': within_cluster_var,
            'percent_variance_between_clusters': percent_between,
            'percent_variance_within_clusters': percent_within,
            'variance_ratio': between_cluster_var / within_cluster_var if within_cluster_var > 0 else np.inf
        }
    
    def visualize_variance_analysis(self, figsize=(20, 15)):
        """Create comprehensive visualizations for variance analysis."""
        clusters = sorted(self.anime_data['cluster'].unique())
        n_clusters = len(clusters)
        
        fig, axes = plt.subplots(3, 3, figsize=figsize)
        fig.suptitle('Anime Cluster Rating Variance Analysis', fontsize=20, fontweight='bold')
        
        # 1. Box plot of all ratings by cluster
        ax1 = axes[0, 0]
        cluster_rating_data = []
        cluster_labels = []
        for cluster in clusters:
            cluster_ratings = self.expanded_data[self.expanded_data['cluster'] == cluster]['rating']
            cluster_rating_data.append(cluster_ratings.values)
            cluster_labels.append(f'Cluster {cluster}')
        
        ax1.boxplot(cluster_rating_data, labels=cluster_labels)
        ax1.set_title('Rating Distributions by Cluster', fontweight='bold')
        ax1.set_ylabel('Rating')
        ax1.grid(True, alpha=0.3)
        
        # 2. Violin plot
        ax2 = axes[0, 1]
        sns.violinplot(data=self.expanded_data, x='cluster', y='rating', ax=ax2)
        ax2.set_title('Rating Density by Cluster', fontweight='bold')
        ax2.set_xlabel('Cluster')
        ax2.set_ylabel('Rating')
        
        # 3. Variance comparison
        ax3 = axes[0, 2]
        cluster_stats = self.cluster_level_analysis()
        ax3.bar(cluster_stats['cluster'].astype(str), cluster_stats['variance_rating'])
        ax3.set_title('Rating Variance by Cluster', fontweight='bold')
        ax3.set_xlabel('Cluster')
        ax3.set_ylabel('Variance')
        
        # 4. Mean vs Variance scatter
        ax4 = axes[1, 0]
        ax4.scatter(cluster_stats['mean_rating'], cluster_stats['variance_rating'], 
                   s=cluster_stats['num_anime']*10, alpha=0.7)
        for i, row in cluster_stats.iterrows():
            ax4.annotate(f'C{row["cluster"]}', 
                        (row['mean_rating'], row['variance_rating']),
                        xytext=(5, 5), textcoords='offset points')
        ax4.set_title('Cluster Mean vs Variance\n(bubble size = # anime)', fontweight='bold')
        ax4.set_xlabel('Mean Rating')
        ax4.set_ylabel('Variance')
        ax4.grid(True, alpha=0.3)
        
        # 5. Rating distribution heatmap
        ax5 = axes[1, 1]
        heatmap_data = []
        for cluster in clusters:
            cluster_data = self.expanded_data[self.expanded_data['cluster'] == cluster]
            dist = [len(cluster_data[cluster_data['rating'] == r]) for r in range(1, 11)]
            total = sum(dist)
            dist_pct = [d/total*100 if total > 0 else 0 for d in dist]
            heatmap_data.append(dist_pct)
        
        sns.heatmap(heatmap_data, 
                   xticklabels=[f'{i}★' for i in range(1, 11)],
                   yticklabels=[f'Cluster {c}' for c in clusters],
                   annot=True, fmt='.1f', cmap='YlOrRd', ax=ax5)
        ax5.set_title('Rating Distribution Heatmap (%)', fontweight='bold')
        
        # 6. Coefficient of variation
        ax6 = axes[1, 2]
        ax6.bar(cluster_stats['cluster'].astype(str), cluster_stats['cv_rating'])
        ax6.set_title('Coefficient of Variation by Cluster', fontweight='bold')
        ax6.set_xlabel('Cluster')
        ax6.set_ylabel('CV (%)')
        
        # 7. Anime-level variance distribution
        ax7 = axes[2, 0]
        anime_stats = self.calculate_anime_level_statistics()
        for cluster in clusters:
            cluster_anime = anime_stats[anime_stats['cluster'] == cluster]
            ax7.hist(cluster_anime['variance'], alpha=0.7, 
                    label=f'Cluster {cluster}', bins=20)
        ax7.set_title('Distribution of Anime-Level Variances', fontweight='bold')
        ax7.set_xlabel('Anime Variance')
        ax7.set_ylabel('Frequency')
        ax7.legend()
        
        # 8. Polarization index by cluster
        ax8 = axes[2, 1]
        polarization_data = []
        score_mapping = {
            'score_01_count': 1, 'score_02_count': 2, 'score_03_count': 3, 'score_04_count': 4, 'score_05_count': 5,
            'score_06_count': 6, 'score_07_count': 7, 'score_08_count': 8, 'score_09_count': 9, 'score_10_count': 10
        }
        
        for cluster in clusters:
            cluster_anime = self.anime_data[self.anime_data['cluster'] == cluster]
            polarizations = []
            for _, anime_row in cluster_anime.iterrows():
                high_ratings = sum([anime_row.get(f'score_{r:02d}_count', 0) for r in [8, 9, 10]])
                low_ratings = sum([anime_row.get(f'score_{r:02d}_count', 0) for r in [1, 2, 3]])
                total_ratings = sum([anime_row.get(score_col, 0) for score_col in score_mapping.keys()])
                if total_ratings > 0:
                    polarization = (high_ratings + low_ratings) / total_ratings
                    polarizations.append(polarization)
            polarization_data.append(polarizations)
        
        ax8.boxplot(polarization_data, labels=[f'C{c}' for c in clusters])
        ax8.set_title('Polarization Index by Cluster\n(high + low ratings / total)', fontweight='bold')
        ax8.set_xlabel('Cluster')
        ax8.set_ylabel('Polarization Index')
        
        # 9. Variance decomposition pie chart
        ax9 = axes[2, 2]
        variance_decomp = self.variance_decomposition()
        labels = ['Between Clusters', 'Within Clusters']
        sizes = [variance_decomp['percent_variance_between_clusters'], 
                variance_decomp['percent_variance_within_clusters']]
        colors = ['lightcoral', 'lightskyblue']
        ax9.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
        ax9.set_title('Variance Decomposition', fontweight='bold')
        
        plt.tight_layout()
        return fig
    
    def generate_variance_report(self):
        """Generate comprehensive variance analysis report."""
        print("=" * 90)
        print("ANIME CLUSTER RATING VARIANCE ANALYSIS REPORT")
        print("=" * 90)
        
        # Dataset overview
        print(f"\n📊 DATASET OVERVIEW")
        print("-" * 50)
        total_ratings = self.anime_data[self.score_columns].fillna(0).sum().sum()
        print(f"Total anime analyzed: {len(self.anime_data):,}")
        print(f"Total individual ratings: {total_ratings:,.0f}")
        print(f"Number of clusters: {self.anime_data['cluster'].nunique()}")
        print(f"Average ratings per anime: {total_ratings/len(self.anime_data):.0f}")
        
        # Cluster-level analysis
        print(f"\n📈 CLUSTER-LEVEL VARIANCE ANALYSIS")
        print("-" * 50)
        cluster_stats = self.cluster_level_analysis()
        display_cols = ['cluster', 'num_anime', 'total_ratings', 'mean_rating', 'std_rating', 
                       'variance_rating', 'cv_rating', 'skewness']
        print(cluster_stats[display_cols].round(3).to_string(index=False))
        
        # Variance decomposition
        print(f"\n🔍 VARIANCE DECOMPOSITION")
        print("-" * 50)
        variance_decomp = self.variance_decomposition()
        print(f"Total variance in ratings: {variance_decomp['total_variance']:.4f}")
        print(f"Between-cluster variance: {variance_decomp['between_cluster_variance']:.4f} ({variance_decomp['percent_variance_between_clusters']:.1f}%)")
        print(f"Within-cluster variance: {variance_decomp['within_cluster_variance']:.4f} ({variance_decomp['percent_variance_within_clusters']:.1f}%)")
        print(f"Variance ratio (between/within): {variance_decomp['variance_ratio']:.4f}")
        
        # Statistical tests
        print(f"\n🧮 STATISTICAL TESTS")
        print("-" * 50)
        test_results = self.statistical_tests()
        
        for test_name, results in test_results.items():
            print(f"\n{test_name.replace('_', ' ').title()}:")
            if 'f_statistic' in results:
                print(f"  F-statistic: {results['f_statistic']:.4f}")
            elif 'h_statistic' in results:
                print(f"  H-statistic: {results['h_statistic']:.4f}")
            elif 'chi2_statistic' in results:
                print(f"  Chi2-statistic: {results['chi2_statistic']:.4f}")
            elif 'statistic' in results:
                print(f"  Test statistic: {results['statistic']:.4f}")
            
            print(f"  p-value: {results['p_value']:.4f}")
            print(f"  Result: {results['interpretation']}")
        
        # Key insights
        print(f"\n💡 KEY INSIGHTS")
        print("-" * 50)
        
        # Most and least variable clusters
        most_variable = cluster_stats.loc[cluster_stats['variance_rating'].idxmax()]
        least_variable = cluster_stats.loc[cluster_stats['variance_rating'].idxmin()]
        
        print(f"Most variable cluster: Cluster {most_variable['cluster']} (variance = {most_variable['variance_rating']:.3f})")
        print(f"Least variable cluster: Cluster {least_variable['cluster']} (variance = {least_variable['variance_rating']:.3f})")
        
        # Clustering effectiveness
        pct_explained = variance_decomp['percent_variance_between_clusters']
        if pct_explained > 20:
            effectiveness = "very effective"
        elif pct_explained > 10:
            effectiveness = "moderately effective"
        elif pct_explained > 5:
            effectiveness = "somewhat effective"
        else:
            effectiveness = "not very effective"
        
        print(f"Clustering effectiveness: {effectiveness} ({pct_explained:.1f}% of variance explained)")
        
        # Rating patterns
        highest_rated = cluster_stats.loc[cluster_stats['mean_rating'].idxmax()]
        lowest_rated = cluster_stats.loc[cluster_stats['mean_rating'].idxmin()]
        
        print(f"Highest-rated cluster: Cluster {highest_rated['cluster']} (mean = {highest_rated['mean_rating']:.2f})")
        print(f"Lowest-rated cluster: Cluster {lowest_rated['cluster']} (mean = {lowest_rated['mean_rating']:.2f})")


# ============================================================================
# RUN VARIANCE ANALYSIS
# ============================================================================

In [23]:
# Create analysis object
if anime_data is not None:
    print("\n🚀 Creating variance analysis object...")
    variance_analyzer = AnimeClusterVarianceAnalysis(anime_data)
    
    print("\n" + "="*80)
    print("📊 RUNNING COMPREHENSIVE VARIANCE ANALYSIS")
    print("="*80)
    
    # Generate comprehensive report
    variance_analyzer.generate_variance_report()
    
else:
    print("❌ Cannot create analyzer - missing required data!")
    variance_analyzer = None


🚀 Creating variance analysis object...
🎯 Variance Analysis initialized:
   Total anime analyzed: 2,000
   Total individual ratings: 351,442,964
   Clusters: 13
   Average ratings per anime: 175721
   Memory-efficient mode: Working with aggregated distributions

📊 RUNNING COMPREHENSIVE VARIANCE ANALYSIS
ANIME CLUSTER RATING VARIANCE ANALYSIS REPORT

📊 DATASET OVERVIEW
--------------------------------------------------
Total anime analyzed: 2,000
Total individual ratings: 351,442,964
Number of clusters: 13
Average ratings per anime: 175721

📈 CLUSTER-LEVEL VARIANCE ANALYSIS
--------------------------------------------------
 cluster  num_anime  total_ratings  mean_rating  std_rating  variance_rating  cv_rating  skewness
       1        245       55746756        7.314       1.710            2.923     23.374    -0.737
       2        127       17083449        8.103       1.496            2.238     18.462    -1.036
       3        165        9055031        6.793       1.849            3.41